## This notebook deals with Joins in PySpark. It deals with the following joins in PySpark:
1. Inner Join
2. Left Join
3. Outer Join
4. Semi and Anti Joins
5. Cross Joins

In [0]:
# Databricks notebook source
from pyspark.sql import SparkSession
from pyspark.sql import Row

# Sample Employee info
data_employee = [
    Row(employee_id=1, name='Alice', city='New York'),
    Row(employee_id=2, name='Bob', city='Los Angeles'),
    Row(employee_id=3, name='Charlie', city='Chicago'),
    Row(employee_id=4, name='David', city='Houston'),
    Row(employee_id=5, name='Eva', city='Phoenix')
]

# Sample Salary info
data_salary = [
    Row(employee_id=1, salary=70000, deprt='HR'),
    Row(employee_id=2, salary=80000, deprt='Engineering'),
    Row(employee_id=3, salary=65000, deprt='Finance'),
    Row(employee_id=6, salary=90000, deprt='Marketing'),
    Row(employee_id=7, salary=75000, deprt='Sales')
]

# spark = SparkSession.getActiveSession()
df_employee = spark.createDataFrame(data_employee)
df_salary = spark.createDataFrame(data_salary)

#Temporary views
df_employee.createOrReplaceTempView("employee")
df_salary.createOrReplaceTempView("salary")

df_employee.display()
df_salary.display()

### **Inner Join**

- Returns only rows that have matching keys in both A and B.
- Returns columns from both A and B.


In [0]:
df_inner=df_employee.join(df_salary, on='employee_id', how='inner')
df_inner.display()

In [0]:
%sql
select 
    e.employee_id, e.name, e.city, s.salary, s.deprt
from employee e inner join salary s 
on e.employee_id=s.employee_id

### Outer Join

An outer join returns all rows from both dataframes, matching rows where possible and filling with nulls where there is no match. This shows all employees and all salary records, even if there is no match.


In [0]:
df_outer=df_employee.join(df_salary, on='employee_id', how='outer')
df_outer.display()

In [0]:
%sql
select 
    e.employee_id, e.name, e.city, s.salary, s.deprt
from employee e 
full outer join 
salary s on
e.employee_id=s.employee_id

### Left Outer Join

- Returns all rows from Table A and matching rows from Table B.
- Unmatched rows from A have NULL values for B's columns.
- Returns columns from both A and B.


In [0]:
df_left_outer=df_employee.join(df_salary, on='employee_id', how='left_outer')
df_left_outer.display()

In [0]:
%sql
select 
    e.employee_id, e.name, e.city, s.salary, s.deprt
from 
employee e 
left outer join 
salary s
on e.employee_id=s.employee_id

### Left Semi Join

- Returns rows from Table A that have a match in Table B.
- Returns columns from Table A only.
- Does not duplicate rows if B has multiple matches.

### Left Anti Join

- Returns rows from Table A that have no match in Table B.
- Returns columns from Table A only.
- Does not duplicate rows.


In [0]:
df_left_semi=df_employee.join(df_salary, on='employee_id', how='left_semi')
df_left_semi.display()
df_left_anti=df_employee.join(df_salary, on='employee_id', how='left_anti')
df_left_anti.display()

In [0]:
%sql
-- select *
-- from employee e
-- left semi join salary s
-- on e.employee_id=s.employee_id
    
select *
from employee e
left anti join salary s
on e.employee_id=s.employee_id

### Cross Join

A cross join returns the Cartesian product of the two dataframes, matching every row in the left dataframe with every row in the right dataframe. Use with caution as it can produce a large number of rows.


In [0]:
df_cross_join=df_employee.join(df_salary, how='cross')
df_cross_join.display()

In [0]:
%sql
select *
from employee e
cross join 
salary s

### Handling Duplicate Rows and Same Column Names in Joins

- Use `dropDuplicates()` to remove duplicate rows from a DataFrame.
- When joining DataFrames, PySpark does not automatically add suffixes to duplicate column names.
- In SQL for removing duplicates, we can use `Distinct` keyword.
- Resolve column name conflicts using:
  - `select()` to choose and rename columns.
  - `withColumnRenamed()` to rename existing columns.

In [0]:
# Add a duplicate row to df_employee for demonstration
df_employee_dup = df_employee.union(df_employee.filter(df_employee.employee_id == 1))
# Remove duplicates
df_employee_nodup = df_employee_dup.dropDuplicates()
# Join and resolve column name conflicts
joined = df_employee_nodup.join(df_salary, on='employee_id', how='inner')
# Rename columns if needed
joined = joined.withColumnRenamed('name', 'employee_name')
display(joined)